# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup (Local)

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected.")


Connected.


## 1. Two paper findings + my methodology questions

Source: `docs/flyrank-seo-research-march-2026.pdf` (*The State of AI-Driven SEO*, March 2026). Constructive audit only — how to make the claims stronger, not a gotcha.

### Finding #1 — Anatomy of Growing Content 

**What they claim:** Rising-impression pages look different from falling ones: longer (≈3.2K vs 2.3K words), younger (≈184 vs 230 days), slightly better position. Large n (≈74K up vs ≈45K down).

**Where does the label/group come from?**  
`up` / `down` from **trend direction**: 30-day impressions vs previous 30 days (>10% up / >10% down). That is a **rule on recent impression change**, not an experiment and not “this page will keep growing.”

**Does the validation design carry the claim?**  
**Partly.** A big observational table supports “growing and declining cohorts **differ** on length/age.” It does **not** by itself support “make pages longer → they will grow.” The paper already tags this as observational / directional — good. To strengthen it: hold client/site constant (grouped comparison), or track the **same** pages before/after an expand-content action (time-aware outcome).

### Finding #3 — Click Capture by Position Tier 

**What they claim:** Weighted CTR falls by position tier (Top 3 ≈0.42% → Deep ≈0.05%). Page-one refinement is a better click bet than spreading effort on deep pages.

**Where does the label/group come from?**  
Groups are **`position_tier` from average position**. Metric is **portfolio weighted CTR** (total clicks ÷ total impressions in the tier) — not a model score, not a per-row average.

**Does the validation design carry the claim?**  
**Yes for the measured pattern** (“CTR is associated with tier in this portfolio”). **Not for causal advice** (“refine snippets → clicks will rise”) unless they later A/B or track revised vs control pages. This finding is the closest cousin of my Lane 4 work: compare CTR **within** tier, don’t treat raw CTR alone as the signal.


## 2. My model under an honest split (before/after)

**Lane 4 — same Week-5 model:** logistic regression ranked by `P(is_ctr_underperformer)`.

**Same slice:** March 2026, `imp_mar >= 100`, real position only.
**Same features:** `imp_mar`, `ctr_mar`, `pos_avg_mar`, `engagement_rate_mar`, `position_tier`.
**Same metric:** Precision@K next to the base rate.

**Why client holdout is the honest one:** pages from one site share niche, CMS, and tracking. A random split can put similar pages in train and test, so the score looks better than it is.

**Same holdout as Week 5:** sort client IDs, then shuffle with `RANDOM_STATE=42` (~75% train / ~25% test clients).

**Why not time-aware here:** features and label are both from March (one snapshot: review now?). Time splits matter later for past-to-future labels.


In [2]:
LABEL = "is_ctr_underperformer"
FEATURE_COLS = ["imp_mar", "ctr_mar", "pos_avg_mar", "engagement_rate_mar", "position_tier"]
RANDOM_STATE = 42


def precision_at_k(scores, labels, k, tie_break=None):
    """Rank by score desc, then tie_break desc (same contract for rule and model)."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels)
    if tie_break is None:
        order = np.argsort(-scores, kind="mergesort")
    else:
        tb = np.asarray(tie_break, dtype=float)
        order = np.lexsort((-tb, -scores))
    k = min(k, len(order))
    return float(labels[order[:k]].mean())


def make_pipeline():
    preprocessor = ColumnTransformer(
        [
            (
                "num",
                StandardScaler(),
                ["imp_mar", "ctr_mar", "pos_avg_mar", "engagement_rate_mar"],
            ),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["position_tier"]),
        ]
    )
    return Pipeline(
        [
            ("prep", preprocessor),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )


def eval_split(train_df, test_df, split_name):
    model = make_pipeline()
    model.fit(train_df[FEATURE_COLS], train_df[LABEL])
    scores = model.predict_proba(test_df[FEATURE_COLS])[:, 1]
    y = test_df[LABEL].to_numpy()
    row = {
        "split": split_name,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "base_rate": float(y.mean()),
    }
    for k in (10, 20, 50):
        row[f"precision_at_{k}"] = precision_at_k(
            scores, y, k, tie_break=test_df["imp_mar"].to_numpy()
        )
    return row, model, scores


# --- Same March feature frame as Week 5 ---
features = con.sql(
    f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar,
            SUM(gsc_clicks) AS clk_mar,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_mar,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_mar,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_mar
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    scored AS (
        SELECT
            *,
            CASE WHEN imp_mar > 0 THEN 100.0 * clk_mar / imp_mar END AS ctr_mar,
            CASE
                WHEN sessions_mar > 0 THEN 100.0 * engaged_mar / sessions_mar
            END AS engagement_rate_mar,
            CASE
                WHEN pos_avg_mar <= 3 THEN 'top_3'
                WHEN pos_avg_mar <= 10 THEN 'page_1'
                WHEN pos_avg_mar <= 20 THEN 'striking'
                WHEN pos_avg_mar <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM daily
        WHERE pos_avg_mar > 0
    ),
    labeled AS (
        SELECT
            s.*,
            MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) AS tier_median_ctr,
            CASE
                WHEN ctr_mar < MEDIAN(ctr_mar) OVER (PARTITION BY position_tier)
                     AND imp_mar >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM scored s
    )
    SELECT * FROM labeled
"""
).df()

model_df = features.dropna(subset=["ctr_mar", "pos_avg_mar"]).copy()
model_df["engagement_rate_mar"] = model_df["engagement_rate_mar"].fillna(0)
model_df["baseline_score"] = model_df["tier_median_ctr"] - model_df["ctr_mar"]
model_df["ctr_gap"] = model_df["baseline_score"]  # label-derived; leakage trap only

# --- BEFORE: random page split ---
train_rand, test_rand = train_test_split(
    model_df, test_size=0.25, random_state=RANDOM_STATE, stratify=model_df[LABEL]
)
row_before, _, _ = eval_split(train_rand, test_rand, "BEFORE: random page split")

# --- AFTER: client holdout (honest) — same split logic as Week 5 ---
rng = np.random.default_rng(RANDOM_STATE)
clients = np.sort(model_df["client_hash_id"].unique())
rng.shuffle(clients)
n_test_clients = max(1, int(round(len(clients) * 0.25)))
test_clients = set(clients[:n_test_clients])

train_client = model_df[~model_df["client_hash_id"].isin(test_clients)].copy()
test_client = model_df[model_df["client_hash_id"].isin(test_clients)].copy()
row_after, model_honest, honest_scores = eval_split(
    train_client, test_client, "AFTER: client holdout"
)
test_client = test_client.assign(model_score=honest_scores)

comparison = pd.DataFrame([row_before, row_after]).set_index("split")
print("Logistic Regression — same features, same metric, two splits:\n")
print(comparison.round(3).to_string())
print(
    f"\nGap (BEFORE − AFTER) Precision@50: "
    f"{row_before['precision_at_50'] - row_after['precision_at_50']:+.3f}"
)
print(
    f"Client holdout base rate: {row_after['base_rate']:.3f} "
    f"(test clients = {len(test_clients)})"
)

OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
split_path = OUT / "validation_split_before_after.csv"
comparison.reset_index().round(3).to_csv(split_path, index=False)
print(f"\nSaved → {split_path}")


Logistic Regression — same features, same metric, two splits:

                           n_train  n_test  base_rate  precision_at_10  precision_at_20  precision_at_50
split                                                                                                   
BEFORE: random page split    76080   25361      0.221              0.6              0.7             0.80
AFTER: client holdout        72470   28971      0.274              0.7              0.8             0.84

Gap (BEFORE − AFTER) Precision@50: -0.040
Client holdout base rate: 0.274 (test clients = 11)

Saved → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\validation_split_before_after.csv


## 3. Leakage audit

**Final feature set:** `imp_mar`, `ctr_mar`, `pos_avg_mar`, `engagement_rate_mar`, `position_tier`

**Attack checklist**

| Risk | Status |
|---|---|
| Label-derived features (`ctr_gap`, `tier_median_ctr`) | Excluded from features — trap below proves why |
| Product flags / health scores | Not in warehouse — not used |
| IDs as features | `client_hash_id` / `content_hash_id` for split/join only |
| Future / overlapping windows | Same-month snapshot only; no `fact_content_query_90d` |
| GA4 zeros | Engagement only when `ga4_data_available IS TRUE`; else filled 0 after filter |

**Trap test** shallow DecisionTree on numeric features, once without `ctr_gap`, once with it. If AUC jumps toward 1.0, the harness works and `ctr_gap` is illegal as a model feature (OK only as the hand baseline score on the `imp_mar >= 500` pool — see ML-07).

**Failures:** top-ranked pages on the client-holdout logistic model that are not underperformers (false alarms).


In [3]:
# --- Leakage trap (same as ML-04): shallow tree, numeric feats ± ctr_gap ---
honest_num = ["imp_mar", "ctr_mar", "pos_avg_mar", "engagement_rate_mar"]
X_honest = model_df[honest_num]
y_all = model_df[LABEL]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_honest, y_all, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)

tree_honest = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
tree_honest.fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, tree_honest.predict_proba(X_te)[:, 1])

X_leaky = X_honest.assign(ctr_gap=model_df["ctr_gap"])
tree_leaky = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
tree_leaky.fit(X_leaky.loc[X_tr.index], y_tr)
auc_leaky = roc_auc_score(y_te, tree_leaky.predict_proba(X_leaky.loc[X_te.index])[:, 1])

print("LEAKAGE TRAP (DecisionTree max_depth=2, same hunt as ML-04)")
print(f"Honest ROC AUC (no ctr_gap):  {auc_honest:.3f}")
print(f"Leaky ROC AUC (with ctr_gap): {auc_leaky:.3f}")
print(
    "Verdict: "
    + (
        "ctr_gap leaks the label — keep it OUT of model features "
        "(OK only as the hand baseline score)."
        if auc_leaky - auc_honest > 0.02
        else "still exclude ctr_gap — it is computed from the label ingredients."
    )
)
print()

# --- Failure examples from the Week-5 logistic model (client holdout) ---
fail = test_client[test_client[LABEL] == 0].nlargest(5, "model_score")[
    [
        "content_hash_id",
        "imp_mar",
        "ctr_mar",
        "position_tier",
        "baseline_score",
        "model_score",
        LABEL,
    ]
]
print("Top-5 FALSE ALARMS on client holdout (high model score, label=0):")
print(fail.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print(
    "Reading: high-visibility pages the model ranks as urgent, "
    "but they are NOT below-tier-median underperformers (label=0). "
    "A reviewer opening them first would waste time — classic false alarm cost."
)

trap = {
    "auc_honest": float(auc_honest),
    "auc_leaky": float(auc_leaky),
    "n_false_alarm_examples": int(len(fail)),
}
trap_path = OUT / "validation_leakage_trap.json"
trap_path.write_text(json.dumps(trap, indent=2))
print(f"\nSaved → {trap_path}")


LEAKAGE TRAP (DecisionTree max_depth=2, same hunt as ML-04)
Honest ROC AUC (no ctr_gap):  0.921
Leaky ROC AUC (with ctr_gap): 1.000
Verdict: ctr_gap leaks the label — keep it OUT of model features (OK only as the hand baseline score).

Top-5 FALSE ALARMS on client holdout (high model score, label=0):
         content_hash_id     imp_mar  ctr_mar position_tier  baseline_score  model_score  is_ctr_underperformer
content_7172a7fad43f0998 205867.0000   0.4187        page_1         -0.2233       0.9999                      0
content_f107e54b10b43725 195997.0000   0.5082        page_1         -0.3127       0.9998                      0
content_f352b7cfd0b2f434 136098.0000   0.2109        page_1         -0.0154       0.9993                      0
content_3b6e4c8d9a0a5c9c 121458.0000   0.3227        page_1         -0.1273       0.9957                      0
content_e0ca055423cbe896  86319.0000   0.3116         top_3         -0.0591       0.9829                      0

Reading: high-visibility 

## 4. Claim rewrite

**Bold (overreaches):**
> Our model beats the baseline and finds the pages that need a CTR fix.

Problems: "beats" without naming the split/metric; "need a CTR fix" sounds causal — we never tested that fixing the page raises clicks.

**Safe rewrite:**
> On March 2026 pages, under a **client holdout**, logistic regression **measured** Precision@K above the base rate (see numbers in the cell below). A random page split can look different — we report the **grouped** number when claiming skill. Week-5 **observed** the model surfaces high-stake underperformers that the fixed ML-07 `ctr_gap` rule ranks hundreds of places lower — even though the rule hits perfect Precision@K on the `imp_mar >= 500` pool by design. This is **decision-support** ranking, not proof that editing a page will raise clicks.

**Also safe to say:** always print base rate next to Precision@K; never claim Google outcomes or causal CTR lifts.


In [4]:
# Receipts for the claim rewrite — honest split numbers only
claim = {
    "bold_claim": "Our model beats the baseline and finds the pages that need a CTR fix.",
    "safe_claim": (
        "On March 2026 pages, under a client holdout, logistic regression measured "
        "Precision@K above the base rate (see numbers in the cell below). A random page "
        "split can look different — we report the grouped number when claiming skill. "
        "Week-5 observed the model surfaces high-stake underperformers that the fixed "
        "ML-07 ctr_gap rule ranks hundreds of places lower — even though the rule hits "
        "perfect Precision@K on the imp_mar >= 500 pool by design. This is "
        "decision-support ranking, not proof that editing a page will raise clicks."
    ),
    "honest_split": row_after,
    "random_split_for_gap_only": row_before,
}
print("BOLD:")
print(" ", claim["bold_claim"])
print()
print("SAFE:")
print(" ", claim["safe_claim"])
print()
print("Honest metrics (AFTER: client holdout):")
for k, v in row_after.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")

print("\nRandom-split metrics (BEFORE — for the gap only):")
for k, v in row_before.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")

def _split_row(row):
    return {
        k: (
            float(v)
            if isinstance(v, (float, np.floating))
            else int(v)
            if isinstance(v, (int, np.integer))
            else v
        )
        for k, v in row.items()
    }


payload = {
    "bold_claim": claim["bold_claim"],
    "safe_claim": claim["safe_claim"],
    "honest_split": _split_row(row_after),
    "random_split_for_gap_only": _split_row(row_before),
}
claim_path = OUT / "validation_claim_rewrite.json"
claim_path.write_text(json.dumps(payload, indent=2))
print(f"\nSaved → {claim_path}")


BOLD:
  Our model beats the baseline and finds the pages that need a CTR fix.

SAFE:
  On March 2026 pages, under a client holdout, logistic regression measured Precision@K above the base rate (see numbers in the cell below). A random page split can look different — we report the grouped number when claiming skill. Week-5 observed the model surfaces high-stake underperformers that the fixed ML-07 ctr_gap rule ranks hundreds of places lower — even though the rule hits perfect Precision@K on the imp_mar >= 500 pool by design. This is decision-support ranking, not proof that editing a page will raise clicks.

Honest metrics (AFTER: client holdout):
  split: AFTER: client holdout
  n_train: 72470
  n_test: 28971
  base_rate: 0.274
  precision_at_10: 0.700
  precision_at_20: 0.800
  precision_at_50: 0.840

Random-split metrics (BEFORE — for the gap only):
  split: BEFORE: random page split
  n_train: 76080
  n_test: 25361
  base_rate: 0.221
  precision_at_10: 0.600
  precision_at_20: 0.700


## Self-check

Before you submit, confirm each line honestly:

- [X] Section 1 (paper findings) filled
- [X] Sections 2–4 filled — markdown thinking AND the code that backs them
- [X] The notebook runs top to bottom with no errors (Runtime → Run all) 
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
